# box2boxcam — preparing the `v001` dataset

This notebook turns the Infactory audit into a first clean, reproducible dataset. It never modifies `data/raw`.

Goals:

1. classify each frame by annotation quality;
2. isolate the cases that need a human decision;
3. create train/validation/test by **complete sequence**;
4. materialize `data/processed/v001`;
5. verify the result automatically.

## 1. Cleaning policy

For this first version, we prefer to lose a few doubtful images rather than teach the model mistakes.

| Status | Rule | `v001` decision |
|---|---|---|
| `positive_single` | exactly one valid box | included |
| `negative` | no label file or empty label | included with an empty label |
| `review_invalid` | malformed line or zero-area box | excluded, review queue |
| `review_multi` | several valid ball boxes | excluded, review queue |

This rule is deliberately conservative. A future `v002` can bring back the corrected cases without modifying `v001`.

In [ ]:
import csv
import ctypes
import json
import os
import random
import shutil
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from box2boxcam.data.audit_yolo import audit

SOURCE_ROOT = PROJECT_ROOT / "data/raw/infactory_ball_tracking/data"
PROCESSED_ROOT = PROJECT_ROOT / "data/processed/v001"
SOURCE_MANIFEST = PROJECT_ROOT / "data/manifests/infactory_ball_tracking.json"
DERIVED_MANIFEST = PROJECT_ROOT / "data/manifests/infactory_v001.json"
REPORT_PATH = PROJECT_ROOT / "models/reports/infactory_v001_summary.json"

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset Infactory absent : {SOURCE_ROOT}")

print("Source :", SOURCE_ROOT)
print("Destination :", PROCESSED_ROOT)

## 2. Classify all frames

We re-read the annotations from the source. The name before the last `_` is the sequence UUID; the final number is the frame's temporal index.

In [ ]:
def inspect_label(image_path):
    label_path = image_path.with_suffix(".txt")
    if not label_path.exists():
        return "negative", 0, []

    rows = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    if not rows:
        return "negative", 0, []

    valid_boxes = []
    invalid_rows = 0
    for row in rows:
        fields = row.split()
        if len(fields) != 5:
            invalid_rows += 1
            continue
        try:
            class_id = int(fields[0])
            x, y, width, height = map(float, fields[1:])
        except ValueError:
            invalid_rows += 1
            continue
        if (
            class_id != 0
            or not all(0 <= value <= 1 for value in (x, y, width, height))
            or width == 0
            or height == 0
        ):
            invalid_rows += 1
            continue
        valid_boxes.append((x, y, width, height))

    if invalid_rows:
        return "review_invalid", len(valid_boxes), valid_boxes
    if len(valid_boxes) == 1:
        return "positive_single", 1, valid_boxes
    if len(valid_boxes) > 1:
        return "review_multi", len(valid_boxes), valid_boxes
    return "negative", 0, []

records = []
for image_path in sorted(SOURCE_ROOT.rglob("*.jpg")):
    sequence, frame_text = image_path.stem.rsplit("_", 1)
    status, box_count, boxes = inspect_label(image_path)
    records.append({
        "sequence": sequence,
        "frame": int(frame_text),
        "status": status,
        "box_count": box_count,
        "image_path": image_path,
        "label_path": image_path.with_suffix(".txt"),
        "boxes": boxes,
    })

status_counts = Counter(record["status"] for record in records)
accepted_statuses = {"positive_single", "negative"}
accepted_count = sum(status_counts[status] for status in accepted_statuses)
review_count = len(records) - accepted_count

display(Markdown(f"""
| Status | Images | Decision |
|---|---:|---|
| One valid box | {status_counts['positive_single']:,} | include |
| Negative | {status_counts['negative']:,} | include |
| Invalid annotation | {status_counts['review_invalid']:,} | review |
| Several boxes | {status_counts['review_multi']:,} | review |
| **Total accepted** | **{accepted_count:,}** | `v001` |
| **Total in review** | **{review_count:,}** | outside `v001` |
"""))

assert len(records) == 7152
assert accepted_count == 7014
assert review_count == 138

### Inspecting the review queue

The gallery shows four invalid annotations and four multi-box images. Valid boxes are red; an invalid image with no rectangle only has a zero-area box. This review explains why these frames are excluded rather than corrected automatically.

In [ ]:
def draw_record(record):
    image = Image.open(record["image_path"]).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for x, y, width, height in record["boxes"]:
        left = (x - width / 2) * image.width
        top = (y - height / 2) * image.height
        right = (x + width / 2) * image.width
        bottom = (y + height / 2) * image.height
        drawing.rectangle((left, top, right, bottom), outline="red", width=4)
    return image

def one_example_per_sequence(status, limit=4):
    examples_by_sequence = {}
    for record in records:
        if record["status"] == status:
            examples_by_sequence.setdefault(record["sequence"], record)
    return list(examples_by_sequence.values())[:limit]

review_examples = (
    one_example_per_sequence("review_invalid")
    + one_example_per_sequence("review_multi")
)

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for axis, record in zip(axes.flat, review_examples):
    axis.imshow(draw_record(record))
    axis.set_title(f"{record['status']} — {record['box_count']} valid box(es)")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Build the splits by sequence

We aim for 70% train, 15% validation and 15% test. Since the 54 sequences have very different lengths, simply taking 70% of the UUIDs is not enough.

The algorithm tries 25,000 deterministic partitions and keeps the one that best balances:

- the total number of accepted frames;
- the positive frames;
- the negative frames.

Each UUID stays entirely in a single split.

In [ ]:
sequence_stats = defaultdict(Counter)
for record in records:
    stats = sequence_stats[record["sequence"]]
    stats["all"] += 1
    stats[record["status"]] += 1
    if record["status"] in accepted_statuses:
        stats["accepted"] += 1

sequence_ids = sorted(sequence_stats)
split_sequence_counts = {"train": 38, "val": 8, "test": 8}
target_ratios = {"train": 0.70, "val": 0.15, "test": 0.15}
metrics = ("accepted", "positive_single", "negative")
metric_totals = {
    metric: sum(stats[metric] for stats in sequence_stats.values())
    for metric in metrics
}

def candidate_mapping(order):
    train_end = split_sequence_counts["train"]
    val_end = train_end + split_sequence_counts["val"]
    return {
        **{sequence: "train" for sequence in order[:train_end]},
        **{sequence: "val" for sequence in order[train_end:val_end]},
        **{sequence: "test" for sequence in order[val_end:]},
    }

def split_summary(mapping):
    summary = {split: Counter() for split in target_ratios}
    for sequence, split in mapping.items():
        summary[split]["sequences"] += 1
        summary[split].update(sequence_stats[sequence])
    return summary

def balance_score(mapping):
    summary = split_summary(mapping)
    return sum(
        (summary[split][metric] / metric_totals[metric] - target_ratios[split]) ** 2
        for split in target_ratios
        for metric in metrics
    )

rng = random.Random(42)
best_mapping = None
best_score = float("inf")
for _ in range(25_000):
    order = sequence_ids.copy()
    rng.shuffle(order)
    mapping = candidate_mapping(order)
    score = balance_score(mapping)
    if score < best_score:
        best_mapping = mapping
        best_score = score

for record in records:
    record["split"] = best_mapping[record["sequence"]]

summary = split_summary(best_mapping)
rows = [
    "| Split | Sequences | Accepted | Positives | Negatives | In review |",
    "|---|---:|---:|---:|---:|---:|",
]
for split in ("train", "val", "test"):
    stats = summary[split]
    review = stats["review_invalid"] + stats["review_multi"]
    rows.append(
        f"| {split} | {stats['sequences']} | {stats['accepted']:,} "
        f"({stats['accepted'] / accepted_count:.1%}) | {stats['positive_single']:,} | "
        f"{stats['negative']:,} | {review:,} |"
    )
display(Markdown("\n".join(rows)))

assert len(best_mapping) == 54
assert set(best_mapping.values()) == {"train", "val", "test"}
assert sum(stats["accepted"] for stats in summary.values()) == accepted_count

### Visual check of the balance

The chart compares the theoretical target with the proportions actually obtained. An exact match is impossible with indivisible sequences, but the gap must stay small.

In [ ]:
splits = ["train", "val", "test"]
positions = list(range(len(splits)))
actual_accepted = [summary[split]["accepted"] / accepted_count for split in splits]
actual_positive = [
    summary[split]["positive_single"] / status_counts["positive_single"]
    for split in splits
]
targets = [target_ratios[split] for split in splits]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([x - 0.25 for x in positions], targets, 0.25, label="target")
ax.bar(positions, actual_accepted, 0.25, label="accepted frames")
ax.bar([x + 0.25 for x in positions], actual_positive, 0.25, label="positives")
ax.set_xticks(positions, splits)
ax.set_ylabel("Share of the dataset")
ax.set_title("Balance achieved without splitting sequences")
ax.legend()
plt.show()

## 4. Materialize `data/processed/v001`

The accepted images are created as **APFS clones**: they have an independent inode but initially share the same physical blocks. This avoids copying about 1 GB while protecting `raw` if a copy is modified later. On another file system, the notebook falls back to a regular copy.

The small positive label files are copied normally. For a negative, we create an empty `.txt` file, a convention YOLO understands. The 138 doubtful cases are only recorded in `review_queue.csv`.

The cell is idempotent: if a complete `v001` already exists, it does not overwrite it.

In [ ]:
def relative_to_project(path):
    return str(path.relative_to(PROJECT_ROOT))

if sys.platform == "darwin":
    clonefile = ctypes.CDLL("libSystem.B.dylib", use_errno=True).clonefile
    clonefile.argtypes = [ctypes.c_char_p, ctypes.c_char_p, ctypes.c_int]
    clonefile.restype = ctypes.c_int
else:
    clonefile = None

def clone_or_copy(source, destination):
    if clonefile is None:
        shutil.copy2(source, destination)
        return
    result = clonefile(os.fsencode(source), os.fsencode(destination), 0)
    if result != 0:
        error_number = ctypes.get_errno()
        raise OSError(error_number, os.strerror(error_number), destination)

manifest_fields = [
    "sequence", "frame", "split", "status", "box_count",
    "source_image", "source_label", "output_image", "output_label",
]

if PROCESSED_ROOT.exists() and not (PROCESSED_ROOT / "manifest.csv").exists():
    raise RuntimeError(
        f"Partial destination detected: {PROCESSED_ROOT}. "
        "Manual inspection required; no automatic overwrite."
    )

if not PROCESSED_ROOT.exists():
    for split in ("train", "val", "test"):
        (PROCESSED_ROOT / split / "images").mkdir(parents=True)
        (PROCESSED_ROOT / split / "labels").mkdir(parents=True)

    manifest_rows = []
    for record in records:
        output_image = ""
        output_label = ""
        if record["status"] in accepted_statuses:
            image_destination = (
                PROCESSED_ROOT / record["split"] / "images" / record["image_path"].name
            )
            label_destination = (
                PROCESSED_ROOT / record["split"] / "labels" / record["label_path"].name
            )
            clone_or_copy(record["image_path"], image_destination)
            if record["status"] == "positive_single":
                shutil.copy2(record["label_path"], label_destination)
            else:
                label_destination.write_text("")
            output_image = relative_to_project(image_destination)
            output_label = relative_to_project(label_destination)

        manifest_rows.append({
            "sequence": record["sequence"],
            "frame": record["frame"],
            "split": record["split"],
            "status": record["status"],
            "box_count": record["box_count"],
            "source_image": relative_to_project(record["image_path"]),
            "source_label": (
                relative_to_project(record["label_path"])
                if record["label_path"].exists() else ""
            ),
            "output_image": output_image,
            "output_label": output_label,
        })

    with (PROCESSED_ROOT / "manifest.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=manifest_fields)
        writer.writeheader()
        writer.writerows(manifest_rows)

    with (PROCESSED_ROOT / "review_queue.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=manifest_fields)
        writer.writeheader()
        writer.writerows(
            row for row in manifest_rows if row["status"].startswith("review_")
        )

    (PROCESSED_ROOT / "dataset.yaml").write_text(
        "train: train/images\nval: val/images\ntest: test/images\nnames:\n  0: ball\n"
    )
    print("v001 materialized.")
else:
    print("v001 already exists: no file overwritten, moving on to verification.")

## 5. Record provenance and verify

The small JSON manifest can be versioned on GitHub: it describes how to rebuild `v001` without containing the images. The detailed CSV stays with the local dataset.

In [ ]:
source_provenance = json.loads(SOURCE_MANIFEST.read_text())
split_counts = {
    split: {
        "sequences": summary[split]["sequences"],
        "images": summary[split]["accepted"],
        "positive_images": summary[split]["positive_single"],
        "negative_images": summary[split]["negative"],
    }
    for split in ("train", "val", "test")
}
derived_manifest = {
    "dataset_id": "box2boxcam_infactory_v001",
    "quality_status": "working_dataset_visual_review_required",
    "source_registry_id": source_provenance["registry_id"],
    "source_revision": source_provenance["resolved_revision"],
    "source_url": source_provenance["source_url"],
    "split_unit": "complete UUID sequence",
    "split_target_ratios": target_ratios,
    "split_search_seed": 42,
    "split_search_candidates": 25_000,
    "included_statuses": sorted(accepted_statuses),
    "excluded_statuses": ["review_invalid", "review_multi"],
    "counts": {
        "source_images": len(records),
        "included_images": accepted_count,
        "review_images": review_count,
        "positive_images": status_counts["positive_single"],
        "negative_images": status_counts["negative"],
        "invalid_images": status_counts["review_invalid"],
        "multi_box_images": status_counts["review_multi"],
    },
    "splits": split_counts,
}
rendered_manifest = json.dumps(derived_manifest, indent=2) + "\n"
DERIVED_MANIFEST.write_text(rendered_manifest)
REPORT_PATH.write_text(rendered_manifest)

prepared_audit = audit(PROCESSED_ROOT, target_size=640)
prepared_ball = prepared_audit["class_metrics"][0]

assert prepared_audit["images"] == accepted_count
assert prepared_audit["issues_count"] == 0
assert prepared_ball["boxes"] == status_counts["positive_single"]
assert prepared_ball["multi_box_images_count"] == 0
assert prepared_audit["negative_images"] == status_counts["negative"]

sequence_splits = defaultdict(set)
for record in records:
    sequence_splits[record["sequence"]].add(record["split"])
assert all(len(splits_for_sequence) == 1 for splits_for_sequence in sequence_splits.values())

display(Markdown(f"""
### Verification passed

- **{prepared_audit['images']:,} images** in `v001`;
- **{prepared_ball['boxes']:,} ball boxes**, all unique per image;
- **{prepared_audit['negative_images']:,} negatives**;
- **0 invalid annotations**;
- **0 UUIDs shared between splits**;
- **{review_count} images** kept in the review queue, outside training.
"""))

## 6. Stratified visual quality check

The previous checks prove that the files are consistent, but not that each box really frames a ball. We therefore display **20 images per split**, always selected with seed `42`: four examples for each of the four ball sizes and four negatives.

The size shown is the equivalent diameter of the box after letterboxing to `640 × 640`: `sqrt(width × height)`. It is not the physical size of the ball; it is a simple measure of how hard the case is for the model.

| Group | Equivalent diameter at 640 | Why inspect it |
|---|---:|---|
| `tiny` | `< 8 px` | cases most easily lost when resizing |
| `small` | `8–16 px` | core of the dataset |
| `medium` | `16–32 px` | normally readable ball |
| `large` | `≥ 32 px` | close-ups and camera changes |
| `negative` | no box | check that a visible ball was not missed |

Each title shows the origin (`Infactory`), the split, the sequence and the frame. The selection favors different sequences to avoid looking at twenty almost identical images. The galleries are also saved in `models/reports/`.

In [ ]:
import math

QA_REPORT_PATH = PROJECT_ROOT / "models/reports/infactory_v001_qa.json"
QA_CATEGORY_ORDER = ("tiny", "small", "medium", "large", "negative")
QA_SAMPLES_PER_CATEGORY = 4

def equivalent_diameter_640(record):
    if record["status"] == "negative":
        return None
    with Image.open(record["image_path"]) as image:
        image_width, image_height = image.size
    _, _, box_width, box_height = record["boxes"][0]
    scale = min(640 / image_width, 640 / image_height)
    width_640 = box_width * image_width * scale
    height_640 = box_height * image_height * scale
    return math.sqrt(width_640 * height_640)

def qa_category(record):
    diameter = equivalent_diameter_640(record)
    if diameter is None:
        return "negative"
    if diameter < 8:
        return "tiny"
    if diameter < 16:
        return "small"
    if diameter < 32:
        return "medium"
    return "large"

def diverse_sample(candidates, sample_count, seed):
    # One example per sequence on the first pass, then fill up if needed.
    local_rng = random.Random(seed)
    by_sequence = defaultdict(list)
    for candidate in candidates:
        by_sequence[candidate["sequence"]].append(candidate)
    sequences = sorted(by_sequence)
    local_rng.shuffle(sequences)
    selected = []
    for sequence in sequences:
        selected.append(local_rng.choice(by_sequence[sequence]))
        if len(selected) == sample_count:
            return selected
    remaining = [candidate for candidate in candidates if candidate not in selected]
    local_rng.shuffle(remaining)
    return selected + remaining[: sample_count - len(selected)]

accepted_records = [
    record for record in records if record["status"] in accepted_statuses
]
for record in accepted_records:
    record["qa_category"] = qa_category(record)
    record["diameter_640"] = equivalent_diameter_640(record)

qa_population = {
    split: Counter(
        record["qa_category"]
        for record in accepted_records
        if record["split"] == split
    )
    for split in ("train", "val", "test")
}

qa_samples = {}
for split_index, split in enumerate(("train", "val", "test")):
    fig, axes = plt.subplots(5, 4, figsize=(18, 18))
    split_samples = []
    for category_index, category in enumerate(QA_CATEGORY_ORDER):
        candidates = [
            record
            for record in accepted_records
            if record["split"] == split and record["qa_category"] == category
        ]
        selected = diverse_sample(
            candidates, QA_SAMPLES_PER_CATEGORY, 42 + split_index * 10 + category_index
        )
        assert len(selected) == QA_SAMPLES_PER_CATEGORY
        split_samples.extend(selected)
        for column, record in enumerate(selected):
            axis = axes[category_index, column]
            axis.imshow(draw_record(record))
            diameter = record["diameter_640"]
            difficulty = "no box" if diameter is None else f"{diameter:.1f} px @640"
            axis.set_title(
                f"Infactory · {split} · {category}\n"
                f"seq {record['sequence'][:8]}… · frame {record['frame']} · {difficulty}",
                fontsize=9,
            )
            axis.axis("off")
    qa_samples[split] = split_samples
    fig.suptitle(
        f"box2boxcam v001 — visual check {split} — source: Infactory Ball Tracking",
        fontsize=16,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.98))
    gallery_path = PROJECT_ROOT / f"models/reports/infactory_v001_qa_{split}.jpg"
    fig.savefig(gallery_path, dpi=140, bbox_inches="tight")
    plt.show()

population_rows = [
    "| Split | Tiny | Small | Medium | Large | Negatives |",
    "|---|---:|---:|---:|---:|---:|",
]
for split in ("train", "val", "test"):
    counts = qa_population[split]
    population_rows.append(
        f"| {split} | {counts['tiny']:,} | {counts['small']:,} | "
        f"{counts['medium']:,} | {counts['large']:,} | {counts['negative']:,} |"
    )
display(Markdown("### Population by difficulty level\n\n" + "\n".join(population_rows)))

## 7. Turn the QA into a correction queue

Reading the 60 images invalidated the assumption "a syntactically valid box = a real ball". We saw boxes on a skull, a bright area or the crowd, as well as a goalkeeper holding a ball in an image declared negative. **`v001` must therefore not yet be frozen as ground truth.**

We do not correct these images automatically: a rule such as "remove all large boxes" would also remove genuine close-ups. Instead we build `qa_review_queue.csv` with explainable signals:

- `large_bbox`: equivalent diameter of at least 32 px at 640;
- `single_negative_gap`: an isolated negative frame between two positives;
- `center_jump`: displacement greater than 25% of the normalized diagonal in one frame;
- `area_jump`: area multiplied or divided by more than four in one frame;
- `visual_sample_*`: an error already confirmed in the galleries above.

These rules are only used to **prioritize human review**. An image absent from the queue is not guaranteed to be perfect, and an image present is not necessarily wrong.

In [ ]:
QA_REVIEW_QUEUE_PATH = PROCESSED_ROOT / "qa_review_queue.csv"

# Five confirmed errors observed in the sample. The other cases remain to be decided.
confirmed_visual_findings = {
    "48797f76-4da7-43a7-9d52-9a7cdf9042ef_000015": (
        "visual_sample_false_positive", "box on a skull; ball absent"
    ),
    "143e0618-a3e8-448c-a133-42f97bca8db2_000006": (
        "visual_sample_false_positive", "box in the crowd; ball absent"
    ),
    "5ee600ae-18ce-4307-82e4-8458b2767ee2_000017": (
        "visual_sample_false_positive", "box on a bright area; ball absent"
    ),
    "81926caa-6cb9-4206-b709-60d55623d18a_000027": (
        "visual_sample_false_positive", "box at the left edge; ball absent"
    ),
    "d52fe89a-6572-4129-b726-20eec92ef146_000284": (
        "visual_sample_false_negative", "goalkeeper holding a ball; empty label"
    ),
}

for record in accepted_records:
    record["review_reasons"] = set()
    record["human_finding"] = ""
    if record["diameter_640"] is not None and record["diameter_640"] >= 32:
        record["review_reasons"].add("large_bbox")
    manual = confirmed_visual_findings.get(record["image_path"].stem)
    if manual:
        reason, finding = manual
        record["review_reasons"].add(reason)
        record["human_finding"] = finding

accepted_by_sequence = defaultdict(list)
for record in accepted_records:
    accepted_by_sequence[record["sequence"]].append(record)

for sequence_records in accepted_by_sequence.values():
    by_frame = {record["frame"]: record for record in sequence_records}
    for record in sequence_records:
        frame = record["frame"]
        previous_record = by_frame.get(frame - 1)
        next_record = by_frame.get(frame + 1)
        if (
            record["status"] == "negative"
            and previous_record is not None
            and next_record is not None
            and previous_record["status"] == "positive_single"
            and next_record["status"] == "positive_single"
        ):
            record["review_reasons"].add("single_negative_gap")

        if (
            record["status"] == "positive_single"
            and previous_record is not None
            and previous_record["status"] == "positive_single"
        ):
            x, y, width, height = record["boxes"][0]
            previous_x, previous_y, previous_width, previous_height = (
                previous_record["boxes"][0]
            )
            center_distance = math.hypot(x - previous_x, y - previous_y)
            area_ratio = max(
                (width * height) / (previous_width * previous_height),
                (previous_width * previous_height) / (width * height),
            )
            if center_distance > 0.25:
                record["review_reasons"].add("center_jump")
            if area_ratio > 4:
                record["review_reasons"].add("area_jump")

review_candidates = [
    record for record in accepted_records if record["review_reasons"]
]
review_fields = [
    "sequence", "frame", "split", "current_status",
    "diameter_640", "review_reasons", "human_finding",
    "source_image", "source_label",
]
with QA_REVIEW_QUEUE_PATH.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=review_fields)
    writer.writeheader()
    for record in sorted(
        review_candidates, key=lambda item: (item["split"], item["sequence"], item["frame"])
    ):
        writer.writerow({
            "sequence": record["sequence"],
            "frame": record["frame"],
            "split": record["split"],
            "current_status": record["status"],
            "diameter_640": (
                f"{record['diameter_640']:.3f}"
                if record["diameter_640"] is not None else ""
            ),
            "review_reasons": ";".join(sorted(record["review_reasons"])),
            "human_finding": record["human_finding"],
            "source_image": relative_to_project(record["image_path"]),
            "source_label": (
                relative_to_project(record["label_path"])
                if record["label_path"].exists() else ""
            ),
        })

reason_counts = Counter(
    reason for record in review_candidates for reason in record["review_reasons"]
)
split_review_counts = Counter(record["split"] for record in review_candidates)
display(Markdown(f"""
### Correction queue created

- **{len(review_candidates):,} priority images** out of {accepted_count:,};
- train: **{split_review_counts['train']:,}**;
- validation: **{split_review_counts['val']:,}**;
- test: **{split_review_counts['test']:,}**;
- errors already visually confirmed: **{len(confirmed_visual_findings)}**.

The CSV states why each frame was selected. It modifies neither `raw` nor the `v001` labels.
"""))

## 8. Independent duplicate check

Splitting by UUID guarantees that a sequence does not cross splits. We additionally compute the SHA-256 of the 7,014 prepared images to verify that no identical file is present under another name in another split. This check detects exact duplicates, not two different frames that look very similar.

In [ ]:
import hashlib

hash_groups = defaultdict(list)
for record in accepted_records:
    prepared_image = (
        PROCESSED_ROOT / record["split"] / "images" / record["image_path"].name
    )
    digest = hashlib.sha256(prepared_image.read_bytes()).hexdigest()
    hash_groups[digest].append(record)

exact_duplicate_groups = [group for group in hash_groups.values() if len(group) > 1]
cross_split_duplicate_groups = [
    group
    for group in exact_duplicate_groups
    if len({record["split"] for record in group}) > 1
]

assert len(hash_groups) == accepted_count
assert not exact_duplicate_groups
assert not cross_split_duplicate_groups

qa_report = {
    "dataset_id": "box2boxcam_infactory_v001",
    "source": "Infactory Ball Tracking",
    "visual_sampling_seed": 42,
    "visual_samples_per_split": len(next(iter(qa_samples.values()))),
    "visual_samples_total": sum(len(sample) for sample in qa_samples.values()),
    "visual_categories": list(QA_CATEGORY_ORDER),
    "population_by_split": {
        split: dict(qa_population[split]) for split in ("train", "val", "test")
    },
    "unique_image_hashes": len(hash_groups),
    "exact_duplicate_groups": len(exact_duplicate_groups),
    "cross_split_duplicate_groups": len(cross_split_duplicate_groups),
    "mechanical_validation": "passed",
    "human_visual_validation": "failed_sample_contains_label_errors",
    "confirmed_visual_errors": len(confirmed_visual_findings),
    "priority_review_images": len(review_candidates),
    "priority_review_by_split": dict(split_review_counts),
    "priority_review_reasons": dict(reason_counts),
}
QA_REPORT_PATH.write_text(json.dumps(qa_report, indent=2) + "\n")

display(Markdown(f"""
### Duplicate check passed

- **{len(hash_groups):,} unique fingerprints** for {accepted_count:,} images;
- **{len(exact_duplicate_groups)} exact duplicates** in the whole dataset;
- **{len(cross_split_duplicate_groups)} exact duplicates** across splits.

The mechanical check is complete, but the visual validation failed: `v001` remains a working version until the correction queue has been processed.
"""))

## Result

`v001` is a **mechanically consistent** baseline, protected against leakage between sequences and checked against exact duplicates. However, the visual sample contains semantic errors: this version remains a working version and must not yet be considered clean ground truth.

The next data action is to process `qa_review_queue.csv` first, starting with the validation/test cases, then produce a `v002` without ever overwriting `raw` or `v001`. The 138 invalid or multi-box annotations from the first review queue will join this correction.

## 9. Build v002 after the human review

Rule decided together: **no decision = keep the original label**, including several real balls. `keep` keeps, `negative` empties the label, `bbox` replaces the boxes with the correction and `exclude` removes the image. The per-sequence exclusions are already recorded as decisions.

The builder re-reads the full manifest (7,152 images), not just the files kept in v001. It keeps the splits by sequence, copies the files into v002 and freezes a copy of the decisions with their SHA-256 fingerprints. Uncorrected invalid annotations block the build. An existing version is never overwritten.

The reusable code in `src/box2boxcam/data/build_v002.py` performs the same build from this notebook or from a command. The report distinguishes annotations accepted by default from explicit decisions; v002 is not presented as fully corrected image by image.

In [ ]:
from box2boxcam.data.build_v002 import build

v002_root = PROJECT_ROOT / 'data/processed/v002'
if v002_root.exists():
    v002_report = json.loads((v002_root / 'build_report.json').read_text())
    print('v002 already exists: reading the frozen report.')
else:
    v002_report = build(PROJECT_ROOT)
display({key: v002_report[key] for key in ('images', 'actions', 'splits', 'quality_status')})